# Day 076 — Exercise 4: run_screen_task

**What you'll build:** `run_screen_task(image, task, analyze_fn=None, llm_fn=None) -> dict` — the two-LLM reasoning pipeline.

**Why it matters:** This is the core of the multimodal agent: vision LLM describes the screen, text LLM reasons about a task using that description.

In [ ]:
from PIL import Image as _PILImage

def _make_mock_image(width=100, height=100, color=(100, 100, 100)):
    return _PILImage.new('RGB', (width, height), color=color)
_mock_analyze_fn    = lambda img, q: 'MOCK:' + q[:16]
_mock_llm_fn        = lambda prompt: 'TASK:' + prompt[:12]
import io, base64

def analyze_screenshot(image, question, analyze_fn=None):
    if analyze_fn is not None:
        return analyze_fn(image, question)
    import ollama
    buf = io.BytesIO()
    image.save(buf, format='PNG')
    img_b64 = base64.b64encode(buf.getvalue()).decode()
    resp = ollama.chat(
        model='llava',
        messages=[{'role': 'user', 'content': question, 'images': [img_b64]}],
    )
    return resp['message']['content']

def describe_screen(image, analyze_fn=None):
    return analyze_screenshot(
        image, 'Describe what you see on this screen in detail.',
        analyze_fn=analyze_fn)

def read_screen_text(image, analyze_fn=None):
    return analyze_screenshot(
        image, 'Extract all visible text from this image exactly as it appears.',
        analyze_fn=analyze_fn)

def find_elements(image, element_type, analyze_fn=None):
    question = (f'List all {element_type} elements visible in this screenshot. '
                'Be specific about their labels, text, or content.')
    return analyze_screenshot(image, question, analyze_fn=analyze_fn)

def answer_about_screen(image, question, analyze_fn=None):
    return analyze_screenshot(image, question, analyze_fn=analyze_fn)


## Task

1. `description = describe_screen(image, analyze_fn=analyze_fn)`
2. Build `context_prompt`: join a list of lines with `'\n'`:
   - `'You are a screen-reading assistant.'`
   - `'Here is what is visible on screen:'`
   - `''` (blank line)
   - `description`
   - `''` (blank line)
   - `f'Task: {task}'`
   - `''` (blank line)
   - `'Answer based only on what is visible on screen.'`
3. If `llm_fn`: `answer = llm_fn(context_prompt)`; else call `ollama.chat` with `llama3.2`
4. Return `{'description': description, 'answer': answer, 'task': task}`

## Your Implementation

In [ ]:
def run_screen_task(image, task, analyze_fn=None, llm_fn=None):
    """Analyze screenshot with vision LLM, then reason with text LLM.

    Returns:
        dict with keys: description, answer, task
    """
    raise NotImplementedError


In [ ]:
def run_screen_task(image, task, analyze_fn=None, llm_fn=None):
    description = describe_screen(image, analyze_fn=analyze_fn)
    lines = [
        'You are a screen-reading assistant.',
        'Here is what is visible on screen:',
        '',
        description,
        '',
        f'Task: {task}',
        '',
        'Answer based only on what is visible on screen.',
    ]
    context_prompt = '\n'.join(lines)
    if llm_fn is not None:
        answer = llm_fn(context_prompt)
    else:
        import ollama
        resp = ollama.chat(
            model='llama3.2',
            messages=[{'role': 'user', 'content': context_prompt}],
        )
        answer = resp['message']['content']
    return {'description': description, 'answer': answer, 'task': task}


## Automated checks

In [ ]:

score, total = 0, 5
try:
    from PIL import Image as PILImage
    img = PILImage.new('RGB', (100, 100))

    result = run_screen_task(img, 'Find title',
                             analyze_fn=_mock_analyze_fn,
                             llm_fn=_mock_llm_fn)
    assert isinstance(result, dict), f"expected dict, got {type(result)}"
    score += 1; print("✅ returns dict")

    assert all(k in result for k in ('description', 'answer', 'task'))
    score += 1; print("✅ dict has description, answer, task keys")

    assert result['task'] == 'Find title'
    score += 1; print("✅ task is preserved in result")

    called = {}
    def _analyze(i, q): called['q'] = q; return 'SCREEN_DESC'
    r2 = run_screen_task(img, 'My task', analyze_fn=_analyze, llm_fn=lambda p: 'DONE')
    assert r2['description'] == 'SCREEN_DESC'
    score += 1; print("✅ description comes from describe_screen (analyze_fn used)")

    prompts = []
    def _llm(p): prompts.append(p); return 'ANSWER'
    r3 = run_screen_task(img, 'Task', analyze_fn=lambda i, q: 'VIS', llm_fn=_llm)
    assert r3['answer'] == 'ANSWER' and 'VIS' in prompts[0]
    score += 1; print("✅ answer from llm_fn; visual description in prompt")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def run_screen_task(image, task, analyze_fn=None, llm_fn=None):
    description = describe_screen(image, analyze_fn=analyze_fn)
    lines = [
        'You are a screen-reading assistant.',
        'Here is what is visible on screen:',
        '',
        description,
        '',
        f'Task: {task}',
        '',
        'Answer based only on what is visible on screen.',
    ]
    context_prompt = '\n'.join(lines)
    if llm_fn is not None:
        answer = llm_fn(context_prompt)
    else:
        import ollama
        resp = ollama.chat(
            model='llama3.2',
            messages=[{'role': 'user', 'content': context_prompt}],
        )
        answer = resp['message']['content']
    return {'description': description, 'answer': answer, 'task': task}
```

**Why join a list of lines?** Building the prompt as a list avoids backslash newline escape sequences inside string literals. Each line is a clear, readable element.

</details>